<a href="https://colab.research.google.com/github/pullz6/Large-Scale-Fraud-Detection-System/blob/main/Data_preprocessing_Model_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mlflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 733.8/733.8 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 9.3 MB/s eta 0:00:00


In [2]:
!pip install --upgrade mlflow>=2.3.0

In [3]:
!pip install pyngrok -q

In [4]:
def set_mlflow():
  get_ipython().system_raw("mlflow ui --port 5000 &")
  ngrok.kill()
  NGROK_AUTH_TOKEN = "your code"
  ngrok.set_auth_token(NGROK_AUTH_TOKEN)
  ngrok_tunnel = ngrok.connect(addr="5000", proto="http", bind_tls=True)
  print("MLflow Tracking UI:", ngrok_tunnel.public_url)
  return ngrok_tunnel.public_url

In [5]:
!pip install -q kaggle

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn import preprocessing

import os
from google.colab import drive
from kaggle.api.kaggle_api_extended import KaggleApi
import json
from pyngrok import ngrok, conf
import mlflow
from datetime import datetime, timedelta

## Load Data from Kaggle

In [8]:
#Read creds
drive.mount('/content/drive')

#Get the creds for Kaggle for the Kaggle api
creds_path = ('/content/drive/MyDrive/Projects/Creds/kaggle.json')

Mounted at /content/drive


In [9]:
with open(creds_path, 'r') as f:
    creds = json.load(f)

In [10]:
!mkdir -p ~/.kaggle

In [11]:
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(creds, f)

In [12]:
# Set permissions
!chmod 600 ~/.kaggle/kaggle.json

# Verify setup
!kaggle datasets list -s "ieee fraud"

# STEP 2: DOWNLOAD DATASET
# Download the dataset (this is a known mirror of the original competition data)
!kaggle datasets download -d kartik2112/fraud-detection

# Unzip the files (creates a 'fraud-detection' folder)
!unzip fraud-detection.zip -d fraud-detection

ref                                                        title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------  -------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
mlg-ulb/creditcardfraud                                    Credit Card Fraud Detection                          69155672  2018-03-23 01:17:27.913000         901902      12240  0.85294116       
muhakabartay/yourallmodelsdata                             IEEE-CIS Fraud Detection Models Data                 29768007  2019-09-18 07:57:04.473000            684         17  1.0              
whenamancodes/fraud-detection                              Fraud Detection                                      69155672  2022-09-12 11:54:40.550000          10600        117  1.0              
kyakovlev/ieee-submissions-and

In [13]:
# List downloaded files
!ls -la fraud-detection/

total 489856
drwxr-xr-x 2 root root      4096 Jun 15 20:02 .
drwxr-xr-x 1 root root      4096 Jun 15 20:02 ..
-rw-r--r-- 1 root root 150354339 Aug  5  2020 fraudTest.csv
-rw-r--r-- 1 root root 351238196 Aug  5  2020 fraudTrain.csv


## Load for training from pandas

In [14]:
train = pd.read_csv("fraud-detection/fraudTrain.csv")
test = pd.read_csv("fraud-detection/fraudTest.csv")

In [15]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long              

In [16]:
def data_prep(data):
  #All the steps to data training
  data.drop(['Unnamed: 0','cc_num','first','last','street','zip','trans_num','lat','long','merchant','merch_lat','merch_long','gender','unix_time','category','city'],inplace=True,axis=1)

  data['trans_date_trans_time']= pd.to_datetime(data['trans_date_trans_time'])
  data['month'] = data['trans_date_trans_time'].dt.month
  data['month'] = data['month'].astype(int)
  data['dob']= pd.to_datetime(data['dob'])
  data['age'] = (data['trans_date_trans_time'] - data['dob']).dt.days/365
  data["job"]= data["job"].astype(str)

  data.drop(['trans_date_trans_time','dob'],inplace=True,axis=1)

  label_encoder = preprocessing.LabelEncoder()
  data["job"]= label_encoder.fit_transform(data["job"])
  data["state"]= label_encoder.fit_transform(data["state"])

  return data

In [17]:
train = data_prep(train)

In [18]:
test['is_fraud'].value_counts()

,count
is_fraud,
0,553574
1,2145


In [19]:
print("Class Distribution:")
print(train['is_fraud'].value_counts())
print(f"Percentage: \n{train['is_fraud'].value_counts(normalize=True) * 100}")

# Imbalance ratio (%)
imbalance_ratio = train['is_fraud'].value_counts().min() / train['is_fraud'].value_counts().max()
print(f"Imbalance ratio: {imbalance_ratio:.3f}")

Class Distribution:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64
Percentage: 
is_fraud
0    99.421135
1     0.578865
Name: proportion, dtype: float64
Imbalance ratio: 0.006


In [20]:
# Split instances into majority vs minority class/classes
df_majority = train[train['is_fraud'] == 0]
df_minority = train[train['is_fraud'] == 1]

# Undersampling: we keep as many majority instances (n) as minority ones
df_majority_downsampled = df_majority.sample(n=len(df_minority),random_state=42)
df_balanced = pd.concat([df_majority_downsampled, df_minority])

print(f"Original dataset: {len(train)}")
print(f"Balanced dataset: {len(df_balanced)}")

Original dataset: 1296675
Balanced dataset: 15012


In [21]:
df_balanced.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15012 entries, 811778 to 1295733
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   amt       15012 non-null  float64
 1   state     15012 non-null  int64  
 2   city_pop  15012 non-null  int64  
 3   job       15012 non-null  int64  
 4   is_fraud  15012 non-null  int64  
 5   month     15012 non-null  int64  
 6   age       15012 non-null  float64
dtypes: float64(2), int64(5)
memory usage: 938.2 KB


In [22]:
# Imbalance ratio (%)
imbalance_ratio = df_balanced['is_fraud'].value_counts().min() / df_balanced['is_fraud'].value_counts().max()
print(f"Imbalance ratio: {imbalance_ratio:.3f}")

Imbalance ratio: 1.000


## Model Development

In [23]:
uri = set_mlflow()
mlflow.set_tracking_uri(uri=uri)

MLflow Tracking UI: https://ce5a-35-237-229-232.ngrok-free.app


In [24]:
mlflow.set_experiment("Fraud-detection")

2025/06/15 20:04:15 INFO mlflow.tracking.fluent: Experiment with name 'Fraud-detection' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/713714836121938973', creation_time=1750017855360, experiment_id='713714836121938973', last_update_time=1750017855360, lifecycle_stage='active', name='Fraud-detection', tags={}>

In [25]:
df_balanced.head()

,amt,state,city_pop,job,is_fraud,month,age
811778,86.19,4,92043,85,0,12,58.657534
1101718,85.81,24,1019,15,0,4,41.213699
800013,99.30,38,184,444,0,12,74.131507
398945,42.09,26,743,489,0,6,47.186301
207455,4.24,42,87124,486,0,4,32.186301


In [26]:
!pip install xgboost

In [27]:
import xgboost as xgb

from sklearn.metrics import accuracy_score,confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, train_test_split

In [28]:
# Split data into X and y
X = df_balanced.drop(columns=['is_fraud'], axis=1)
y = df_balanced['is_fraud']

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=0)

In [30]:
y_train.value_counts(normalize=True)

,proportion
is_fraud,
0,0.5
1,0.5


In [31]:
dmatrix_train = xgb.DMatrix(data=X_train, label=y_train)
dmatrix_test = xgb.DMatrix(data=X_test,label=y_test)

In [32]:
learning_objective = {'objective':'binary:logistic' }
model = xgb.train(params = learning_objective, dtrain= dmatrix_train)

In [33]:
test_predictions = model.predict(dmatrix_test)
round_test_predictions = [round(p) for p in test_predictions]
accuracy_score(y_test,round_test_predictions)

0.8960923623445826

In [34]:
model_info = mlflow.keras.log_model(
    model=model,
    artifact_path="Model",
    registered_model_name="Model",
    input_example=X_train.iloc[[0]]
)

2025/06/15 20:06:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/15 20:06:38 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


FileNotFoundError: [Errno 2] No such file or directory: '/tmp/tmp15jj4fsk/model/input_example.json'

## Archive

In [39]:
from pyspark.sql import SparkSession

In [40]:
spark = SparkSession.builder.appName("IEEE_Fraud_Detection").config("spark.driver.memory", "8g").config("spark.executor.memory", "8g").getOrCreate()

In [41]:
# Load the training and testing data
test = spark.read.csv(
    f"fraud-detection/fraudTest.csv",
    header=True,
    inferSchema=True
)

train = spark.read.csv(
    f"fraud-detection/fraudTrain.csv",
    header=True,
    inferSchema=True
)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# DataFrame approach
print(f"Training transactions: {train.count():,} rows")
train.printSchema()

# Show some statistics
train.describe().show()

In [ ]:
test.describe().show()

In [ ]:
fake = Faker()

def generate_fake_fraud_data(num_records):
    data = []
    for _ in range(num_records):
        # Generate base transaction data
        cc_num = fake.credit_card_number()
        merchant = fake.company()
        category = random.choice(['entertainment', 'food_dining', 'gas_transport',
                                 'grocery_net', 'grocery_pos', 'health_fitness',
                                 'home', 'kids_pets', 'misc_net', 'misc_pos',
                                 'personal_care', 'shopping_net', 'shopping_pos',
                                 'travel'])
        amt = round(random.uniform(1, 1000), 2)

        # Generate personal information
        gender = random.choice(['M', 'F'])
        first = fake.first_name_male() if gender == 'M' else fake.first_name_female()
        last = fake.last_name()

        # Generate location data
        street = fake.street_address()
        city = fake.city()
        state = fake.state_abbr()
        zip_code = fake.zipcode()
        lat = float(fake.latitude())
        long = float(fake.longitude())
        city_pop = random.randint(1000, 1000000)

        # Generate employment data
        job = fake.job()

        # Transaction identifiers
        trans_num = fake.uuid4()
        unix_time = int(fake.date_time_this_year().timestamp())

        # Merchant location (slightly different from customer)
        merch_lat = lat + random.uniform(-0.1, 0.1)
        merch_long = long + random.uniform(-0.1, 0.1)

        # Fraud determination (5% base fraud rate)
        is_fraud = 0
        if random.random() < 0.05:
            is_fraud = 1
            # Make fraudulent transactions look different
            amt = round(random.uniform(500, 5000), 2)  # Higher amounts
            category = random.choice(['misc_net', 'shopping_net', 'travel'])  # Common fraud categories
            merch_lat = lat + random.uniform(-1, 1)  # More distant from customer
            merch_long = long + random.uniform(-1, 1)

        data.append(Row(
            _c0=len(data) + 1,  # Sequential ID
            cc_num=cc_num,
            merchant=merchant,
            category=category,
            amt=amt,
            first=first,
            last=last,
            gender=gender,
            street=street,
            city=city,
            state=state,
            zip=zip_code,
            lat=lat,
            long=long,
            city_pop=city_pop,
            job=job,
            trans_num=trans_num,
            unix_time=unix_time,
            merch_lat=merch_lat,
            merch_long=merch_long,
            is_fraud=is_fraud
        ))
    return data

# Generate 1000 fake records
fake_data = generate_fake_fraud_data(1000)

# Convert to Spark DataFrame
fake_df = spark.createDataFrame(fake_data)
fake_df.show(5)

In [ ]:
train = train.unionByName(fake_df, allowMissingColumns=True)

In [ ]:
train.describe().show()

In [ ]:
train.printSchema()

In [ ]:
uri = set_mlflow()
mlflow.set_tracking_uri(uri=uri)

In [74]:
train = pd.concat([train, fake_df])